# NB7 - Detector diagnostics and calibration  (run once, re-run whenever the detector is retrained)
Answers three questions with numbers instead of guesses, and writes the answers to one small file that the engines notebook reads.

1. **What does the detector actually see in my image?** (every raw detection, with the threshold it had to pass)
2. **How good is it by object size?** (recall for each class for small / medium / large objects)
3. **Which settings should the engines use?**
   * the **machine threshold** and the detector input size (960 vs 1280), picked on the detector's own validation split with a recall-oriented score (F2)
   * whether a second, open-vocabulary machine detector (YOLO-World) helps
   * the **"no helmet / vest box found" prior** by head size: how often a worker with *no* PPE detection really has no PPE. This replaces the flat 0.5 in the PPE engine. It also gives the head size below which PPE is **unverifiable**.

## Expects (Google Drive)
```
MyDrive/cv5570_risk/
   weights/   det_ppe_worker_best.pt, class_thresholds_ppe_worker.json        <- fine-tuning notebook
   data/      <dataset>.zip   the dataset the detector was TRAINED on (the Roboflow construction-site-safety one with data.yaml and the
              10 classes Hardhat, NO-Hardhat, Person, machinery, ...). NOT the MOCS zip: that one only has worker boxes.
   demo/images/   (optional) your own images for the raw-detection viewer
```
## Produces
```
MyDrive/cv5570_risk/
   weights/risk_config.json          <- read by the engines notebook (v3)
   diagnostics/*.csv, *.png          <- the tables and plots, for the report
```
The tuning split is **valid** (the detector's thresholds were also fitted there, so a little optimism is possible). **Report final numbers on the test split**: set `SPLIT = 'test'` and re-run only to produce the tables (do not overwrite `risk_config.json`; `WRITE_CONFIG` below controls that).

**Limits to state honestly:** the "missing PPE" rate comes from that dataset, where the share of workers without PPE may differ from your site photos; head size is estimated from the person box (13% of its height), so it is approximate.

In [ ]:
!pip -q install "ultralytics>=8.4.0" pandas
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, glob, json, math, random, zipfile, datetime
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT        = '/content/drive/MyDrive/cv5570_risk'
DATASET     = None      # zip (or folder) of the dataset the detector was trained on. None = auto: the only zip in data/ that is not the MOCS one
DET_WEIGHTS = f'{ROOT}/weights/det_ppe_worker_best.pt'
THR_JSON    = f'{ROOT}/weights/class_thresholds_ppe_worker.json'
CONFIG_OUT  = f'{ROOT}/weights/risk_config.json'
DIAG_DIR    = f'{ROOT}/diagnostics'
os.makedirs(DIAG_DIR, exist_ok=True)

SPLIT        = 'valid'          # tuning split. Use 'test' only to produce report tables (and set WRITE_CONFIG = False)
WRITE_CONFIG = True
MAX_IMGS     = 300              # images to evaluate (None = all)
SIZES        = (960, 1280)      # detector input sizes to compare
EVAL_WORLD   = True             # also test the open-vocabulary machine detector (downloads yolov8s-worldv2.pt, needs internet)
WORLD_MODEL  = 'yolov8s-worldv2.pt'
WORLD_PROMPTS = ['concrete pump truck', 'crane', 'excavator', 'dump truck', 'loader', 'bulldozer', 'truck', 'construction machine']
HEAD_BINS    = [0, 10, 16, 24, 36, 10000]       # head height in pixels AT DETECTOR RESOLUTION (see markdown at the end)
VIEW_IMAGES  = sorted(glob.glob(f'{ROOT}/demo/images/*'))[:6]     # images for the raw-detection viewer
MACHINE_NAMES = ('machinery', 'vehicle')

# the settings the engines will use (hand-set; edit here). Written to risk_config.json by the cell right after the models load.
MANUAL = dict(
    det_imgsz=1280,
    machine_thr=0.05,
    person_thr=0.25,
    world=dict(use=True, thr=0.30, model=WORLD_MODEL,
               prompts=WORLD_PROMPTS),
    # placeholders until the dataset evaluation below replaces them with measured values (WRITE_MEASURED_PRIORS)
    head_bins=[0, 10, 16, 24, 36, 1e9], prior_helmet=[0.5, 0.6, 0.7, 0.7, 0.7], prior_vest=[0.5, 0.6, 0.7, 0.7, 0.7],
    unverifiable_below_px=10.0, band_edges=None,
)
WRITE_MEASURED_PRIORS = True    # after the dataset evaluation: replace the placeholder priors / unverifiable size with the measured ones

In [ ]:
# LOAD MODELS
det = YOLO(DET_WEIGHTS)
THR = json.load(open(THR_JSON))
NAMES = det.names                                   # {id: name} of the detector
print('detector classes:', list(NAMES.values()))
print('current per-class thresholds:', THR)
assert all(m in NAMES.values() for m in MACHINE_NAMES), 'machine class names differ - edit MACHINE_NAMES'

In [ ]:

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0

def nms(items, thr=0.5):
    """items: dicts with 'box' and 'conf' -> kept list (class-agnostic)"""
    keep = []
    for d in sorted(items, key=lambda d: -d['conf']):
        if all(iou(d['box'], k['box']) < thr for k in keep): keep.append(d)
    return keep

def count_matches(preds, gts, thr=0.5):
    """one-to-one greedy matching, highest IoU first -> true positives"""
    pairs = sorted(((iou(a, b), i, j) for i, a in enumerate(preds) for j, b in enumerate(gts)), reverse=True)
    up, ug, tp = set(), set(), 0
    for v, i, j in pairs:
        if v < thr: break
        if i in up or j in ug: continue
        up.add(i); ug.add(j); tp += 1
    return tp

def read_gt(lbl_path, W, H):
    """YOLO label file (boxes or polygons) -> [(class name, [x1, y1, x2, y2])] in pixels"""
    out = []
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            q = line.split()
            if len(q) < 5: continue
            v = list(map(float, q[1:]))
            if len(v) == 4: x, y, w, h = v; x0, x1, y0, y1 = x - w / 2, x + w / 2, y - h / 2, y + h / 2
            else: xs, ys = v[0::2], v[1::2]; x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            out.append((GT_NAMES[int(q[0])], [x0 * W, y0 * H, x1 * W, y1 * H]))
    return out

def fbeta(tp, npred, ngt, beta=2.0):
    P = tp / max(1, npred); R = tp / max(1, ngt)
    return (1 + beta ** 2) * P * R / max(1e-9, beta ** 2 * P + R), P, R

## 0. Write the settings the engines will use
Nothing below this cell is needed to run the engines: this cell alone writes `weights/risk_config.json` with the `MANUAL` settings from the config cell. The sections after it are diagnostics (viewer, evaluation on the training dataset).

In [ ]:
old = json.load(open(CONFIG_OUT)) if os.path.exists(CONFIG_OUT) else {}
manual_cfg = dict(MANUAL, created=str(datetime.date.today()), source='manual settings from NB7 config cell')
if old.get('band_edges'): manual_cfg['band_edges'] = old['band_edges']
os.makedirs(os.path.dirname(CONFIG_OUT), exist_ok=True)
json.dump(manual_cfg, open(CONFIG_OUT, 'w'), indent=1)
print('saved', CONFIG_OUT); print(json.dumps({k: v for k, v in manual_cfg.items() if k != 'world'}, indent=1)); print('world:', {k: v for k, v in manual_cfg['world'].items() if k != 'prompts'})

## 1. Raw detections for your own images
Every detection the model makes at confidence 0.05 or more. **Green** = passes the class threshold the pipeline currently uses; **red** = seen but filtered out. If a machine only shows up in red, a lower threshold is the fix; if it does not show up at all, the model has not learned that kind of machine and the open-vocabulary detector (orange, if `EVAL_WORLD`) is the fix.

In [ ]:
_world = None
def get_world():
    global _world
    if _world is None:
        from ultralytics import YOLOWorld
        _world = YOLOWorld(WORLD_MODEL); _world.set_classes(WORLD_PROMPTS)
    return _world

def world_dets(img, conf=0.05, imgsz=960):
    r = get_world().predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
    return [dict(name='world:' + WORLD_PROMPTS[int(b.cls)], conf=float(b.conf), box=[float(v) for v in b.xyxy[0]]) for b in r.boxes]

def det_all(img, imgsz, conf=0.05):
    r = det.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
    return [dict(name=NAMES[int(b.cls)], conf=float(b.conf), box=[float(v) for v in b.xyxy[0]]) for b in r.boxes]

def raw_view(path, imgsz=1280, with_world=EVAL_WORLD):
    img = cv2.imread(path); vis = img.copy(); fs = max(0.5, img.shape[1] / 1600); th = max(1, int(img.shape[1] / 800))
    rows = []
    for d in det_all(img, imgsz):
        ok = d['conf'] >= THR.get(d['name'], 0.25); col = (60, 180, 75) if ok else (0, 0, 230)
        x1, y1, x2, y2 = [int(v) for v in d['box']]
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, th)
        cv2.putText(vis, f"{d['name']} {d['conf']:.2f}", (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, fs * 0.6, col, max(1, th - 1))
        rows.append(dict(source='fine-tuned', cls=d['name'], conf=round(d['conf'], 2), threshold=THR.get(d['name']), passes=ok))
    if with_world:
        try:
            for d in world_dets(img):
                x1, y1, x2, y2 = [int(v) for v in d['box']]
                cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 140, 255), th)
                cv2.putText(vis, f"{d['name'][6:]} {d['conf']:.2f}", (x1, min(img.shape[0] - 4, y2 + 14)), cv2.FONT_HERSHEY_SIMPLEX, fs * 0.6, (0, 140, 255), max(1, th - 1))
                rows.append(dict(source='YOLO-World', cls=d['name'][6:], conf=round(d['conf'], 2), threshold=None, passes=None))
        except Exception as e:
            print('YOLO-World not available:', e)
    plt.figure(figsize=(11, 8)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title(os.path.basename(path)); plt.show()
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(['source', 'conf'], ascending=[True, False])
        machines = df[df['cls'].isin(list(MACHINE_NAMES))]
        print('machine detections (fine-tuned):', 'none at all' if machines.empty else f"best conf {machines.conf.max()} vs threshold {[THR.get(m) for m in MACHINE_NAMES]}")
        display(df.head(25))

for p in VIEW_IMAGES: raw_view(p)
if not VIEW_IMAGES: print('no images in demo/images - put some there (or set VIEW_IMAGES) to use the viewer')

## 1b. Load the detector's own validation data (needs the training dataset zip in `data/`)

In [ ]:
DS_RAW = '/content/ppe_dataset'
if not os.path.isdir(DS_RAW):
    src = DATASET
    if src is None:
        cands = [z for z in glob.glob(f'{ROOT}/data/*.zip') if 'mocs' not in os.path.basename(z).lower()]
        assert len(cands) == 1, f'put the detector\'s training dataset zip in {ROOT}/data/ or set DATASET above (found: {cands})'
        src = cands[0]
    print('dataset:', src)
    if os.path.isdir(src): os.symlink(src, DS_RAW)
    else: zipfile.ZipFile(src).extractall(DS_RAW)
yamls = glob.glob(f'{DS_RAW}/**/data.yaml', recursive=True)
assert yamls, 'no data.yaml found in the dataset - this notebook needs the dataset with the 10 detector classes'
import yaml
_n = yaml.safe_load(open(yamls[0]))['names']
GT_NAMES = {int(k): v for k, v in (_n.items() if isinstance(_n, dict) else enumerate(_n))}          # class id -> name, from the dataset itself
print('dataset classes:', list(GT_NAMES.values()))
assert set(GT_NAMES.values()) >= {'Person', 'Hardhat', 'NO-Hardhat', 'machinery'}, (
    'this dataset does not have the detector classes (Person, Hardhat, NO-Hardhat, machinery...). The MOCS zip only has workers; use the dataset the detector was trained on.')
ds_root = os.path.dirname(yamls[0])
split_dir = next((f'{ds_root}/{s}' for s in (SPLIT, 'valid', 'val', 'test') if os.path.isdir(f'{ds_root}/{s}/images')), None)
assert split_dir, 'no images folder found in the dataset'
print('using split folder:', split_dir)

## 2. Run the detector once on the validation split (both input sizes)

In [ ]:
imgs = sorted(glob.glob(f'{split_dir}/images/*.*'))
if MAX_IMGS: random.Random(0).shuffle(imgs); imgs = imgs[:MAX_IMGS]
GT, DET, SHAPE = {}, {s: {} for s in SIZES}, {}
for p in imgs:
    img = cv2.imread(p)
    if img is None: continue
    H, W = img.shape[:2]; SHAPE[p] = (H, W)
    GT[p] = read_gt(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt', W, H)
    for s in SIZES: DET[s][p] = det_all(img, s)
print(len(GT), 'images;', sum(len(v) for v in GT.values()), 'ground-truth boxes')
gt_count = pd.Series([n for v in GT.values() for n, _ in v]).value_counts(); display(gt_count.to_frame('boxes in ground truth').T)
assert gt_count.get('Person', 0) > 0 and gt_count.get('Hardhat', 0) > 0 and sum(gt_count.get(m, 0) for m in MACHINE_NAMES) > 0, (
    'the ground truth is missing Person / Hardhat / machine boxes - wrong dataset? (see the table above)')

## 3. Recall by object size (is the PPE problem resolution?)
Object size is its height in pixels **at the detector's input resolution**. `recall @ threshold` is what the pipeline gets now; `recall @ 0.05` is the most any threshold could get. A big gap means the threshold is throwing away real detections; a low value even at 0.05 means the model does not see that object at that size.

In [ ]:
SIZE_BINS = [(0, 24, '<24 px'), (24, 48, '24-48 px'), (48, 96, '48-96 px'), (96, 1e9, '>=96 px')]
CLASSES = ['Person', 'Hardhat', 'NO-Hardhat', 'Safety Vest', 'NO-Safety Vest', 'machine']
def is_cls(name, c): return name in MACHINE_NAMES if c == 'machine' else name == c

def size_table(size):
    rows = []
    for c in CLASSES:
        for lo, hi, lab in SIZE_BINS:
            n = tp_t = tp_l = 0
            for p in GT:
                H, W = SHAPE[p]; k = size / max(H, W)
                gts = [(b, (b[3] - b[1]) * k) for nm, b in GT[p] if is_cls(nm, c)]
                gts = [b for b, h in gts if lo <= h < hi]
                if not gts: continue
                dets = [d for d in DET[size][p] if is_cls(d['name'], c)]
                dt = [d['box'] for d in dets if d['conf'] >= (min(THR.get(m, 0.25) for m in MACHINE_NAMES) if c == 'machine' else THR.get(c, 0.25))]
                dl = [d['box'] for d in dets]
                n += len(gts); tp_t += count_matches(dt, gts); tp_l += count_matches(dl, gts)
            if n: rows.append(dict(cls=c, size=lab, n_gt=n, recall_at_threshold=round(tp_t / n, 3), recall_at_0_05=round(tp_l / n, 3)))
    return pd.DataFrame(rows)

for s in SIZES:
    t = size_table(s); t.to_csv(f'{DIAG_DIR}/recall_by_size_{s}.csv', index=False)
    print(f'--- detector input {s}'); display(t.pivot(index='cls', columns='size', values='recall_at_threshold').reindex(columns=[b[2] for b in SIZE_BINS]).add_prefix('recall@thr '))

## 4. Machine detection: threshold and input size (merged `machinery` + `vehicle`)
Machines are merged into one class and scored with **F2** (recall counts twice as much as precision: missing a machine costs more than a false alarm). "current" is what the pipeline did before: each class at its own F1 threshold.

In [ ]:
def machine_gt(p): return [b for nm, b in GT[p] if nm in MACHINE_NAMES]
def machine_pred(p, size, thr):
    return [d['box'] for d in nms([d for d in DET[size][p] if d['name'] in MACHINE_NAMES and d['conf'] >= thr])]

def eval_machine(pred_fn):
    tp = npred = ngt = 0
    for p in GT:
        pr, g = pred_fn(p), machine_gt(p); tp += count_matches(pr, g); npred += len(pr); ngt += len(g)
    return fbeta(tp, npred, ngt)

GRID = [0.05, 0.08, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6]
sweep = []
for s in SIZES:
    for t in GRID:
        f2, P, R = eval_machine(lambda p: machine_pred(p, s, t)); sweep.append(dict(size=s, thr=t, F2=round(f2, 3), precision=round(P, 3), recall=round(R, 3)))
    # what the pipeline did before: each machine class at its own threshold
    cur = lambda p, s=s: [d['box'] for d in nms([d for d in DET[s][p] if d['name'] in MACHINE_NAMES and d['conf'] >= THR.get(d['name'], 0.25)])]
    f2, P, R = eval_machine(cur); sweep.append(dict(size=s, thr='current', F2=round(f2, 3), precision=round(P, 3), recall=round(R, 3)))
sw = pd.DataFrame(sweep); sw.to_csv(f'{DIAG_DIR}/machine_sweep.csv', index=False); display(sw)
num = sw[sw.thr != 'current']; best = num.loc[num.F2.idxmax()]
assert best.F2 > 0, 'machine F2 is 0 at every threshold - the ground truth or class names are wrong; nothing will be saved'
DET_IMGSZ, MACHINE_THR = int(best['size']), float(best['thr'])
print(f'-> best: input {DET_IMGSZ}, machine threshold {MACHINE_THR}  (F2 {best.F2}, precision {best.precision}, recall {best.recall})')
plt.figure(figsize=(6, 4))
for s in SIZES:
    d = num[num['size'] == s]; plt.plot(d.recall, d.precision, 'o-', label=f'input {s}')
plt.xlabel('recall'); plt.ylabel('precision'); plt.title('machine detection (merged)'); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f'{DIAG_DIR}/machine_pr.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# open-vocabulary second machine detector (optional): world alone, then "fine-tuned OR world"
WORLD = dict(use=False, thr=None)
if EVAL_WORLD:
    try:
        W_DET = {}
        for p in GT: W_DET[p] = world_dets(cv2.imread(p), conf=0.05, imgsz=960)
        ws = []
        for t in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]:
            f2, P, R = eval_machine(lambda p, t=t: [d['box'] for d in nms([d for d in W_DET[p] if d['conf'] >= t])])
            ws.append(dict(thr=t, F2=round(f2, 3), precision=round(P, 3), recall=round(R, 3)))
        ws = pd.DataFrame(ws); display(ws); wbest = ws.loc[ws.F2.idxmax()]
        union = lambda p: [d['box'] for d in nms([dict(box=b, conf=1.0) for b in machine_pred(p, DET_IMGSZ, MACHINE_THR)] +
                                                  [d for d in W_DET[p] if d['conf'] >= wbest['thr']])]
        f2u, Pu, Ru = eval_machine(union)
        print(f"fine-tuned alone: F2 {best.F2} (P {best.precision}, R {best.recall})")
        print(f"YOLO-World alone (thr {wbest['thr']}): F2 {wbest.F2} (P {wbest.precision}, R {wbest.recall})")
        print(f"fine-tuned OR YOLO-World: F2 {f2u:.3f} (P {Pu:.3f}, R {Ru:.3f})")
        WORLD = dict(use=bool(f2u >= best.F2 + 0.02), thr=float(wbest['thr']))
        ws.to_csv(f'{DIAG_DIR}/world_sweep.csv', index=False)
        print('-> use YOLO-World as a second machine detector:', WORLD['use'])
        print('   (the validation machines are in the fine-tuned model\'s own domain, so this comparison is the hard case for YOLO-World;'
              ' it is meant for machine types the training data does not cover, such as the pump truck and rail crane. Check it in the viewer above on those images.)')
    except Exception as e:
        print('YOLO-World evaluation skipped:', e)

## 5. "No PPE box found": how often is the PPE really missing? (by head size)
For every ground-truth worker whose PPE status is labelled in the dataset (a Hardhat or NO-Hardhat box on the head; a Safety Vest or NO-Safety Vest box on the torso), we check what the detector would have said, using the same rules as the engine. For workers with **no helmet or no-helmet box found**, `prior_missing` is the share that really have no helmet (with light smoothing). That is the number that replaces the flat 0.5. `recall_when_wearing` is how often a worn helmet or vest is found at all: when it is low, the worker's PPE is **unverifiable** at that head size.

Head size = 13% of the person box height, converted to pixels at detector resolution (`x input size / longer image side`).

In [ ]:
ZONE = dict(helmet=(0.0, 0.40), vest=(0.10, 0.75))
def in_zone(box, person, zone):
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2; x1, y1, x2, y2 = person; h = max(1.0, y2 - y1)
    return x1 <= cx <= x2 and y1 <= cy <= y2 and zone[0] <= (cy - y1) / h <= zone[1]

def head_bin(px): return next(i for i in range(len(HEAD_BINS) - 1) if HEAD_BINS[i] <= px < HEAD_BINS[i + 1])

recs = []
for p in GT:
    H, W = SHAPE[p]; k = DET_IMGSZ / max(H, W)
    people = [b for nm, b in GT[p] if nm == 'Person']
    for pb in people:
        head_px = 0.13 * (pb[3] - pb[1]) * k
        row = dict(bin=head_bin(head_px), head_px=head_px)
        for item, pos, neg, zn in (('helmet', 'Hardhat', 'NO-Hardhat', ZONE['helmet']), ('vest', 'Safety Vest', 'NO-Safety Vest', ZONE['vest'])):
            g_pos = any(nm == pos and in_zone(b, pb, zn) for nm, b in GT[p]); g_neg = any(nm == neg and in_zone(b, pb, zn) for nm, b in GT[p])
            status = None if (g_pos == g_neg) else ('wearing' if g_pos else 'missing')          # unlabelled or contradictory -> skipped
            d_pos = any(d['name'] == pos and d['conf'] >= THR.get(pos, 0.25) and in_zone(d['box'], pb, zn) for d in DET[DET_IMGSZ][p])
            d_neg = any(d['name'] == neg and d['conf'] >= THR.get(neg, 0.25) and in_zone(d['box'], pb, zn) for d in DET[DET_IMGSZ][p])
            row[item] = dict(status=status, found_pos=d_pos, found_neg=d_neg)
        recs.append(row)
print(len(recs), 'ground-truth workers')

def prior_table(item):
    rows = []
    for b in range(len(HEAD_BINS) - 1):
        rs = [r for r in recs if r['bin'] == b and r[item]['status'] is not None]
        wear = [r for r in rs if r[item]['status'] == 'wearing']; miss = [r for r in rs if r[item]['status'] == 'missing']
        noev = [r for r in rs if not r[item]['found_pos'] and not r[item]['found_neg']]
        n_noev_missing = sum(r[item]['status'] == 'missing' for r in noev)
        neg_found = [r for r in rs if r[item]['found_neg']]
        rows.append(dict(item=item, head_px=f'{HEAD_BINS[b]}-{HEAD_BINS[b+1] if HEAD_BINS[b+1] < 1e4 else "inf"}', n_labelled=len(rs),
                         share_missing=round(len(miss) / max(1, len(rs)), 3),
                         recall_when_wearing=round(sum(r[item]['found_pos'] for r in wear) / max(1, len(wear)), 3) if wear else None,
                         n_no_evidence=len(noev),
                         prior_missing=round((n_noev_missing + 1) / (len(noev) + 2), 3),                  # Laplace smoothing toward 0.5
                         precision_of_NO_box=round(sum(r[item]['status'] == 'missing' for r in neg_found) / max(1, len(neg_found)), 3) if neg_found else None))
    return pd.DataFrame(rows)

pt = pd.concat([prior_table('helmet'), prior_table('vest')]); pt.to_csv(f'{DIAG_DIR}/ppe_prior_by_head_size.csv', index=False); display(pt)

In [ ]:
helm, vest = pt[pt.item == 'helmet'].reset_index(drop=True), pt[pt.item == 'vest'].reset_index(drop=True)
MIN_RECALL = 0.35
ok = [i for i, r in helm.iterrows() if r.recall_when_wearing is not None and r.recall_when_wearing >= MIN_RECALL and r.n_labelled >= 10]
UNVERIFIABLE_BELOW = float(HEAD_BINS[ok[0]]) if ok else 0.0
if not ok: print('WARNING: no head-size bin reaches helmet recall', MIN_RECALL, '- nothing is marked unverifiable; check the table above')
print(f'-> PPE is "unverifiable" when the head is smaller than {UNVERIFIABLE_BELOW:.0f} px at detector resolution')
print('-> prior(helmet missing | no box found) per head-size bin:', list(helm.prior_missing))
print('-> prior(vest missing | no box found)   per head-size bin:', list(vest.prior_missing))
print('\nREAD THIS: if these priors come out well below 0.5, then "nothing detected" really is mostly "worn but missed" in this dataset, and the engine will'
      ' (correctly, for this dataset) score such workers lower. Photos with many bare-headed workers will have a higher true rate; you can override'
      ' the values in risk_config.json by hand (keys prior_helmet / prior_vest).')

## 6. Save the measured PPE priors
Only the measured head-size priors and the unverifiable size are written (when `WRITE_MEASURED_PRIORS`). The machine threshold, input size, person threshold and YOLO-World settings stay as set in `MANUAL`; the sweep above is for reporting and for choosing them next time.

In [ ]:
meas = dict(head_bins=HEAD_BINS[:-1] + [1e9], prior_helmet=[float(x) for x in helm.prior_missing], prior_vest=[float(x) for x in vest.prior_missing],
            unverifiable_below_px=UNVERIFIABLE_BELOW)
print('measured:', json.dumps(meas)); print('sweep suggestion (not applied):', dict(det_imgsz=DET_IMGSZ, machine_thr=MACHINE_THR, world=WORLD))
if WRITE_MEASURED_PRIORS and WRITE_CONFIG:
    rc = json.load(open(CONFIG_OUT)); rc.update(meas); rc['tuned_on'] = dict(split=os.path.basename(split_dir), n_images=len(GT))
    json.dump(rc, open(CONFIG_OUT, 'w'), indent=1); print('priors saved to', CONFIG_OUT)
else:
    print('not saved: placeholder priors from MANUAL stay in risk_config.json')